# Paso 1 -- Verificar las 4 fórmulas de recompensa en un paso de ejemplo

Antes de entrenar nada: confirmar que las 4 fórmulas (A, B, C, D -- ver
`modelo_rl/prueba_rewards/README.md`) se calculan bien, sobre el MISMO par
de estados (antes/después de un paso concreto), y que dan números
coherentes con lo esperado de cada una. También un chequeo de sanity
matemático: con `eta_potencial=0`, C debe coincidir EXACTAMENTE con B
(el término de shaping se anula para cualquier par de estados cuando
`eta=0`, no solo cuando los estados son iguales -- ver la fórmula de
`Phi(s)=-eta*N(s)` en el README).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

BASE_DIR = Path.cwd().parent
SIMULADOR_DIR = (BASE_DIR / "../../simulador").resolve()
POLITICA_BASE_DIR = (BASE_DIR / "../../politica_base").resolve()
MODELO_RL_DIR = (BASE_DIR / "..").resolve()
BB_DIR = (BASE_DIR / "../../bergen-boats").resolve()
DEMAND_DIR = (BASE_DIR / "../../demand").resolve()
sys.path.insert(0, str(DEMAND_DIR / "src"))
sys.path.insert(0, str(SIMULADOR_DIR / "src"))
sys.path.insert(0, str(POLITICA_BASE_DIR / "src"))
sys.path.insert(0, str(MODELO_RL_DIR / "src"))
sys.path.insert(0, str(BASE_DIR / "src"))

import masas as demand_masas
import llegadas as demand_llegadas
from politica_base import asignar_flota
from entrenamiento import EntornoDemandaAleatoria
from recompensas_alternativas import calcular_recompensa_A, calcular_recompensa_B, calcular_recompensa_C, calcular_recompensa_D, potencial

cfg_propio = yaml.safe_load(open(BASE_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_sim = yaml.safe_load(open(SIMULADOR_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_bb = yaml.safe_load(open(BB_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_demand = demand_masas.cargar_config(DEMAND_DIR / "config" / "instance.yaml")

resumen_masas = pd.read_csv(DEMAND_DIR / "output" / "masas_por_nodo.csv", index_col="id")
intensidad_od = pd.read_csv(DEMAND_DIR / "output" / "matriz_intensidad_od.csv")
poblacion_total_zonas = resumen_masas["poblacion_total"].sum()
conexiones_fuertes = cfg_bb["garantia"]["conexiones_fuertes"]
nodos = [n["id"] for n in cfg_bb["nodos_demanda"]]
matriz_tiempos = pd.read_csv(BB_DIR / "02_ruteo_navegable" / "output" / "matriz_tiempos_min.csv", index_col=0)

cfg_recompensas = cfg_propio["recompensas"]
cfg_entren_propio = cfg_propio["entrenamiento"]
cfg_agente = cfg_sim["agente"]
cfg_hiper = cfg_agente["hiperparametros"]
cfg_escalon = cfg_sim["escalones"][cfg_entren_propio["escalon_base"]]
hora_ini_min, hora_fin_min = cfg_escalon["horas"][0] * 60, cfg_escalon["horas"][1] * 60
semillas_eval = cfg_agente["evaluacion"]["semillas"]
gamma = cfg_agente["gamma"]

print("Recompensas definidas:", list(cfg_recompensas.keys()))
for tipo, c in cfg_recompensas.items():
    nombre = c["nombre"]
    print(f"  {tipo}: {nombre}")
escalon_base = cfg_entren_propio["escalon_base"]
num_barcos, horas = cfg_escalon["num_barcos"], cfg_escalon["horas"]
print(f"\nInstancia: {escalon_base} ({num_barcos} barcos, {horas}h)")
timesteps = cfg_entren_propio["total_timesteps"]
print(f"Timesteps de entrenamiento (cada una): {timesteps}")
print(f"Semillas de evaluacion: {semillas_eval}")


def construir_entorno_base():
    return EntornoDemandaAleatoria(
        generar_llegadas_dia_fn=demand_llegadas.generar_llegadas_dia,
        cfg_demand=cfg_demand, intensidad_od=intensidad_od, conexiones_fuertes=conexiones_fuertes,
        poblacion_total_zonas=poblacion_total_zonas, horas=cfg_escalon["horas"],
        porcentaje_poblacion_dia=cfg_escalon["porcentaje_poblacion_dia"],
        matriz_tiempos=matriz_tiempos, nodos=nodos, num_barcos=cfg_escalon["num_barcos"],
        capacidad_barco=cfg_bb["flota"]["capacidad_pasajeros"], nodo_inicial=cfg_bb["flota"]["nodo_inicial"],
        paso_tiempo_min=cfg_sim["paso_tiempo_min"], hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_sim["recompensa"], unidad_demanda=cfg_sim["unidad_demanda"],
    )


Recompensas definidas: ['A', 'B', 'C', 'D']
  A: Objetivo puro (identidad de Little)
  B: Ponderada convexa con tolerancia, sin techo
  C: B + potential-based shaping (Ng, Harada & Russell 1999)
  D: sin tolerancia y 0.001 movimiento (parametros de produccion)

Instancia: escalon_1 (2 barcos, [6, 9]h)
Timesteps de entrenamiento (cada una): 150000
Semillas de evaluacion: [1001, 1002, 1003, 1004, 1005]


## Construir un par de estados (antes/después) no trivial

Se corre la política base unos pasos (semilla fija, reproducible) para que
haya gente esperando de verdad -- en el primer paso de una corrida las
colas están vacías y las 4 fórmulas darían 0, poco informativo.

In [2]:
env = construir_entorno_base()
obs, info = env.reset(seed=1001)

N_PASOS_PREVIOS = 15
for _ in range(N_PASOS_PREVIOS):
    estado = info["_estado_obj"]
    libres = [b for b in estado.barcos if b.libre]
    decisiones = asignar_flota(libres, estado, matriz_tiempos, env.capacidad_barco, cfg_sim["recompensa"])
    accion = np.array([env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0 for b in estado.barcos])
    obs, r, terminated, truncated, info = env.step(accion)

# El paso de ejemplo: uno mas, con estado ANTES y DESPUES capturados aparte.
estado_antes = info["_estado_obj"]
libres = [b for b in estado_antes.barcos if b.libre]
decisiones = asignar_flota(libres, estado_antes, matriz_tiempos, env.capacidad_barco, cfg_sim["recompensa"])
accion = np.array([env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0 for b in estado_antes.barcos])
obs, r_original, terminated, truncated, info = env.step(accion)
estado_despues = info["_estado_obj"]

n_antes = sum(u.tamano for cola in estado_antes.colas.values() for u in cola) + sum(len(b.a_bordo) for b in estado_antes.barcos)
n_despues = sum(u.tamano for cola in estado_despues.colas.values() for u in cola) + sum(len(b.a_bordo) for b in estado_despues.barcos)
print(f"Minuto del paso de ejemplo: {estado_despues.t_actual_min}")
print(f"Personas activas ANTES del paso: {n_antes}")
print(f"Personas activas DESPUES del paso: {n_despues}")


Minuto del paso de ejemplo: 392.0
Personas activas ANTES del paso: 27
Personas activas DESPUES del paso: 27


## Las 4 fórmulas, mismo paso, lado a lado

In [3]:
paso_tiempo_min = cfg_sim["paso_tiempo_min"]

r_A, desglose_A = calcular_recompensa_A(estado_despues, cfg_recompensas["A"], paso_tiempo_min)
r_B, desglose_B = calcular_recompensa_B(estado_despues, cfg_recompensas["B"])
r_C, desglose_C = calcular_recompensa_C(estado_antes, estado_despues, cfg_recompensas["C"], gamma)
r_D, desglose_D = calcular_recompensa_D(estado_despues, cfg_recompensas["D"])

filas = []
for tipo, r, desglose in [("A", r_A, desglose_A), ("B", r_B, desglose_B), ("C", r_C, desglose_C), ("D", r_D, desglose_D)]:
    fila = {"tipo": tipo, "nombre": cfg_recompensas[tipo]["nombre"], "r": r}
    fila.update(desglose)
    filas.append(fila)

tabla_ejemplo = pd.DataFrame(filas)
display(tabla_ejemplo.round(4))
print("\nComo referencia, el reward que hubiera devuelto env.py con la formula de produccion (ignorado, se recalcula):", round(r_original, 4))


,tipo,nombre,r,tiempo_activo,movimiento,entrega,incomodidad,shaping
0,A,Objetivo puro (identidad de Little),-54.0120,54.0,0.012,0.0,NaN,NaN
1,B,"Ponderada convexa con tolerancia, sin techo",-2.3155,NaN,0.006,0.0,2.3095,NaN
2,C,"B + potential-based shaping (Ng, Harada & Russ...",-2.3020,NaN,0.006,0.0,2.3095,0.0135
3,D,sin tolerancia y 0.001 movimiento (parametros ...,-7.6907,NaN,0.002,0.0,7.6887,NaN



Como referencia, el reward que hubiera devuelto env.py con la formula de produccion (ignorado, se recalcula): -2.5095


**Lectura esperada:** A es la más chica en magnitud por persona pero escala con
`tamaño x 2 min` de TODOS los activos (no solo los atrasados) así que puede
ser grande si hay mucha gente en el sistema, aunque nadie esté fuera de
tolerancia todavía. B y D usan la misma fórmula cuadrática, D con techo
bajo (1.0) y `peso_movimiento` mucho más caro (0.1 vs. 0.003 de B) -- si el
paso de ejemplo tiene algún barco moviéndose, D debería penalizar ese
movimiento mucho más fuerte que B. C es B más el término de shaping --
debería diferir de B en exactamente `desglose_C["shaping"]`.

## Chequeo de sanity: con `eta_potencial=0`, C debe coincidir EXACTAMENTE con B

`Phi(s) = -eta * N(s)` -- si `eta=0`, `Phi(s)=0` para CUALQUIER estado, así
que el término de shaping `gamma*Phi(s')-Phi(s)` es 0 sin importar qué tan
distintos sean `estado_antes` y `estado_despues`. No es un caso especial
de "estados iguales", es una propiedad de la fórmula: con `eta=0`, C debe
ser bit-a-bit igual a B, siempre.

In [4]:
cfg_C_eta_cero = dict(cfg_recompensas["C"], eta_potencial=0.0)
r_C_eta_cero, desglose_C_eta_cero = calcular_recompensa_C(estado_antes, estado_despues, cfg_C_eta_cero, gamma)

shaping_eta_cero = desglose_C_eta_cero["shaping"]
print(f"r_B             = {r_B!r}")
print(f"r_C (eta=0)     = {r_C_eta_cero!r}")
print(f"shaping (eta=0) = {shaping_eta_cero!r}")

assert desglose_C_eta_cero["shaping"] == 0.0, "con eta=0 el termino de shaping debe ser exactamente 0"
assert r_C_eta_cero == r_B, "con eta=0, C debe coincidir bit a bit con B"
print("\nOK: con eta_potencial=0, C == B exactamente.")


r_B             = -2.3155466049382696
r_C (eta=0)     = -2.3155466049382696
shaping (eta=0) = 0.0

OK: con eta_potencial=0, C == B exactamente.


## Segundo chequeo: el signo del shaping tiene sentido

Si la cola se achicó en este paso (`estado_despues` tiene menos gente
activa que `estado_antes`), `Phi` subió (menos negativo) y el término de
shaping debería ser POSITIVO -- un premio extra por progresar. Si la cola
creció, debería ser negativo.

In [5]:
eta = cfg_recompensas["C"]["eta_potencial"]
phi_antes = potencial(estado_antes, eta)
phi_despues = potencial(estado_despues, eta)
shaping = gamma * phi_despues - phi_antes

print(f"N(antes)={n_antes}, N(despues)={n_despues}")
print(f"Phi(antes)={phi_antes:.4f}, Phi(despues)={phi_despues:.4f}")
print(f"Termino de shaping = gamma*Phi(despues) - Phi(antes) = {shaping:.4f}")
if n_despues < n_antes:
    print("La cola se achico -> shaping deberia ser positivo:", "OK" if shaping > 0 else "revisar")
elif n_despues > n_antes:
    print("La cola crecio -> shaping deberia ser negativo:", "OK" if shaping < 0 else "revisar")
else:
    print("N no cambio -- shaping = (gamma-1)*Phi; con gamma<1 y Phi<0 (hay gente activa), (gamma-1)*Phi > 0: un premio chico y constante mientras la cola no se mueva, del orden de (1-gamma)*|Phi|")


N(antes)=27, N(despues)=27
Phi(antes)=-1.3500, Phi(despues)=-1.3500
Termino de shaping = gamma*Phi(despues) - Phi(antes) = 0.0135
N no cambio -- shaping = (gamma-1)*Phi; con gamma<1 y Phi<0 (hay gente activa), (gamma-1)*Phi > 0: un premio chico y constante mientras la cola no se mueva, del orden de (1-gamma)*|Phi|
